# Classroom Attendance Prediction: Phase 1 — Data Cleaning & Validation
## Capstone Project: Classroom Attendance Prediction Using Academic Schedule and Historical Attendance Data

### 📌 Project Objective:
The objective of this notebook is to ingest raw physical classroom attendance observations, validate the dataset against the required schema (22 core attributes), check for missing values and anomalies, enforce boundary constraints ($0 \le \text{Students Present} \le \text{Total Enrolled}$), recalculate the attendance percentage to guarantee mathematical integrity, and produce a clean, standardized dataset for modeling.

---
### 🛠️ Key Pipeline Steps:
1. **Automated Path Discovery**: Works seamlessly on both **Kaggle** (`/kaggle/input/...`) and local environments.
2. **Schema & Integrity Validation**: Confirms all 22 required academic schedule attributes are present.
3. **Boundary & Anomaly Correction**: Enforces physical student count constraints.
4. **Mathematical Consistency**: Recalculates $\text{Attendance Percentage} = \frac{\text{Students Present}}{\text{Total Enrolled}} \times 100$.
5. **Cleaned Dataset Export**: Generates `attendance_cleaned.csv` and an audit report.


### 1. Library Imports


In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np

print(f"Python Environment: Pandas {pd.__version__}, NumPy {np.__version__}")


### 2. Kaggle & Local Dataset Auto-Discovery Utility
This helper function automatically searches Kaggle input directories (`/kaggle/input/...`) or standard local repository paths to locate the dataset without requiring hardcoded paths.


In [ ]:
def find_data_file(filename="attendance_raw.csv"):
    """
    Auto-discovers datasets and model artifacts in Kaggle input/working directories
    or local relative repository folders.
    """
    ext = os.path.splitext(filename)[1].lower()

    # 1. Search Kaggle input paths
    kaggle_input = "/kaggle/input"
    if os.path.exists(kaggle_input):
        for root, dirs, files in os.walk(kaggle_input):
            if filename in files:
                p = os.path.join(root, filename)
                print(f"[Kaggle Input] Found: {p}")
                return p
            for f in files:
                if ext and f.lower().endswith(ext) and filename.lower().replace(ext, "") in f.lower():
                    p = os.path.join(root, f)
                    print(f"[Kaggle Input] Found matching file: {p}")
                    return p

    # 2. Search Kaggle working directory
    if os.path.exists("/kaggle/working"):
        p = os.path.join("/kaggle/working", filename)
        if os.path.exists(p):
            print(f"[Kaggle Working] Found: {p}")
            return p
        for root, dirs, files in os.walk("/kaggle/working"):
            if filename in files:
                p = os.path.join(root, filename)
                print(f"[Kaggle Working Tree] Found: {p}")
                return p

    # 3. Search local project paths
    local_candidates = [
        os.path.join("data", "processed", filename),
        os.path.join("..", "data", "processed", filename),
        os.path.join("data", "raw", filename),
        os.path.join("..", "data", "raw", filename),
        os.path.join("models", filename),
        os.path.join("..", "models", filename),
        os.path.join("reports", filename),
        os.path.join("..", "reports", filename),
        filename,
        os.path.join("..", filename)
    ]
    for p in local_candidates:
        if os.path.exists(p):
            print(f"[Local Path] Found: {p}")
            return p

    # 4. Search recursively in current working tree
    for root, dirs, files in os.walk("."):
        if filename in files:
            p = os.path.join(root, filename)
            print(f"[Tree Search] Found: {p}")
            return p

    raise FileNotFoundError(f"Could not find '{filename}'.")

def get_output_dir(subfolder=""):
    """Determines writable output directory (/kaggle/working/ or local folder)."""
    if os.path.exists("/kaggle/working"):
        out_dir = os.path.join("/kaggle/working", subfolder) if subfolder else "/kaggle/working"
    else:
        out_dir = os.path.join("..", subfolder) if os.path.exists("..") else (subfolder if subfolder else ".")
    os.makedirs(out_dir, exist_ok=True)
    return out_dir


### 3. Load and Inspect Raw Dataset


In [ ]:
raw_data_path = find_data_file("attendance_raw.csv")
df_raw = pd.read_csv(raw_data_path)

print(f"Raw Dataset Loaded Successfully!")
print(f"Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns\n")
display(df_raw.head(5))


### 4. Raw Data Inspection & Statistics


In [ ]:
print("=== Dataset Information ===")
df_raw.info()
print("\n=== Missing Value Counts ===")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0] if df_raw.isnull().sum().sum() > 0 else "Zero missing values detected!")


### 5. Schema Validation (22 Core Features)
We verify that the dataset contains all required attributes for academic schedule and attendance tracking.


In [ ]:
REQUIRED_COLUMNS = [
    "Date", "Day of Week", "Lecture Number", "Start Time", "Subject",
    "Faculty ID", "Semester", "Branch", "Section", "Classroom",
    "Total Enrolled Students", "Students Present", "Attendance Percentage",
    "Previous Lecture Attendance", "Gap Since Previous Lecture", "Practical/Theory",
    "Internal Test Week", "Assignment Due", "Holiday Before/After", "Weather",
    "Special Event", "Faculty Experience"
]

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
if missing_cols:
    print(f"[WARN] Missing expected columns: {missing_cols}")
else:
    print("[OK] Schema Validation Passed: All 22 core attributes are present!")


### 6. Data Cleaning & Standardization Logic
We implement the complete cleaning procedure:
1. **Deduplication**: Remove identical repeated entries.
2. **Date Standardization**: Parse dates into ISO `YYYY-MM-DD` and verify `Day of Week`.
3. **Boundary Constraint Validation**: Ensure $0 \le \text{Students Present} \le \text{Total Enrolled Students}$.
4. **Target Variable Integrity**: Mathematically enforce $\text{Attendance Percentage} = \text{round}\left(\frac{\text{Present}}{\text{Enrolled}} \times 100, 2\right)$.
5. **Categorical Normalization**: Clean binary flags (`Yes`/`No`), pedagogical format (`Theory`/`Practical`), and weather categories.


In [ ]:
def clean_attendance_dataset(df_input):
    df = df_input.copy()
    initial_count = len(df)
    
    # 1. Deduplication
    df = df.drop_duplicates().reset_index(drop=True)
    dups_removed = initial_count - len(df)
    
    # 2. Date parsing (support DD-MM-YYYY, YYYY-MM-DD, mixed formats)
    df["Date_Parsed"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=True, errors="coerce")
    df = df.dropna(subset=["Date_Parsed"]).reset_index(drop=True)
    df["Date"] = df["Date_Parsed"].dt.strftime("%Y-%m-%d")
    df["Day of Week"] = df["Date_Parsed"].dt.day_name()
    
    # 3. Numeric conversions & Boundary constraints
    df["Total Enrolled Students"] = pd.to_numeric(df["Total Enrolled Students"], errors="coerce").fillna(60).astype(int)
    df["Total Enrolled Students"] = df["Total Enrolled Students"].clip(lower=1)
    
    df["Students Present"] = pd.to_numeric(df["Students Present"], errors="coerce")
    
    # Impute Students Present if missing but Attendance Percentage is provided
    if df["Students Present"].isna().any() and "Attendance Percentage" in df.columns:
        mask_nan = df["Students Present"].isna()
        df.loc[mask_nan, "Students Present"] = np.round(
            (pd.to_numeric(df.loc[mask_nan, "Attendance Percentage"], errors="coerce") / 100.0) * df.loc[mask_nan, "Total Enrolled Students"]
        )
    
    df["Students Present"] = df["Students Present"].fillna(0).astype(int)
    # Cap present students to total enrolled (physical impossibility safeguard)
    df["Students Present"] = np.clip(df["Students Present"], a_min=0, a_max=df["Total Enrolled Students"])
    
    # 4. Strict Target Integrity Recalculation
    df["Attendance Percentage"] = np.round((df["Students Present"] / df["Total Enrolled Students"]) * 100.0, 2)
    
    # 5. Lecture Number & Semester
    df["Lecture Number"] = pd.to_numeric(df["Lecture Number"], errors="coerce").fillna(1).astype(int).clip(lower=1, upper=12)
    df["Semester"] = pd.to_numeric(df["Semester"], errors="coerce").fillna(1).astype(int).clip(lower=1, upper=8)
    
    # 6. Autoregressive Lags & Gaps
    df["Previous Lecture Attendance"] = pd.to_numeric(df["Previous Lecture Attendance"], errors="coerce")
    df["Previous Lecture Attendance"] = df["Previous Lecture Attendance"].fillna(df["Attendance Percentage"].mean()).clip(0.0, 100.0)
    df["Gap Since Previous Lecture"] = pd.to_numeric(df["Gap Since Previous Lecture"], errors="coerce").fillna(24.0).clip(0.0, 168.0)
    
    # 7. Categorical flags
    def std_binary(val):
        if pd.isna(val): return "No"
        s = str(val).strip().capitalize()
        return "Yes" if s in ["Yes", "Y", "1", "True"] else "No"
    
    for flag_col in ["Internal Test Week", "Assignment Due", "Holiday Before/After", "Special Event"]:
        if flag_col in df.columns:
            df[flag_col] = df[flag_col].apply(std_binary)
        else:
            df[flag_col] = "No"
            
    df["Practical/Theory"] = df["Practical/Theory"].astype(str).str.strip().str.capitalize()
    df["Practical/Theory"] = df["Practical/Theory"].apply(lambda x: "Practical" if "prac" in x.lower() or "lab" in x.lower() else "Theory")
    df["Weather"] = df["Weather"].fillna("Sunny").astype(str).str.strip().str.capitalize()
    df["Faculty Experience"] = pd.to_numeric(df["Faculty Experience"], errors="coerce").fillna(5.0).clip(0.0, 45.0)
    
    # 8. Sort chronologically
    df = df.sort_values(by=["Date_Parsed", "Start Time", "Lecture Number"]).reset_index(drop=True)
    df = df.drop(columns=["Date_Parsed"])
    
    return df, {"initial_records": initial_count, "final_records": len(df), "duplicates_removed": dups_removed}

df_clean, audit = clean_attendance_dataset(df_raw)
print("=== Cleaning Audit Summary ===")
for k, v in audit.items():
    print(f" - {k}: {v}")


### 7. Export Cleaned Dataset for Subsequent Modeling


In [ ]:
out_dir = get_output_dir("data/processed" if not os.path.exists("/kaggle/working") else "")
out_file = os.path.join(out_dir, "attendance_cleaned.csv")
df_clean.to_csv(out_file, index=False)
print(f"[OK] Cleaned dataset successfully saved to: {out_file}")
print(f"Summary: {len(df_clean)} rows, {len(df_clean.columns)} columns")
display(df_clean.head(5))


### 8. Phase 1 Conclusion & Summary:
- Successfully validated schema integrity and resolved missing values / boundary anomalies.
- Guaranteed target variable mathematical precision ($0-100\%$).
- Produced standardized `attendance_cleaned.csv` ready for Exploratory Data Analysis (Phase 2) and Feature Engineering (Phase 3).
